# LeetCode 2062 — Count Vowel Substrings of a String bằng Python

Link đề: https://leetcode.com/problems/count-vowel-substrings-of-a-string/ · Độ khó: **Easy** · Chủ đề: Hash Table, String

```python
class Solution:
    def countVowelSubstrings(self, word: str) -> int:

```

**Mục tiêu học**

1. Hiểu **substring** (chuỗi con liên tiếp) và cách liệt kê mọi substring bằng hai chỉ số `i <= j`, slicing `word[i:j+1]`.
2. Dùng `set` để kiểm tra "chỉ gồm nguyên âm" (`set(sub) <= VOWELS`) và "đủ 5 nguyên âm" (`len(set(sub)) == 5`).
3. Tiến hóa lời giải: `O(n³)` → `O(n²)` (mở rộng dần + **dừng khi gặp phụ âm**) → `O(n)` (**ghi nhớ vị trí xuất hiện cuối** của từng nguyên âm).
4. Biết kỹ thuật "**đúng K loại = tối đa K − tối đa K−1**" với cửa sổ trượt (sliding window).
5. Nộp `class Solution`, đối chiếu các cách bằng test ngẫu nhiên.

## 1. Đề bài yêu cầu gì?

- **Substring**: dãy ký tự **liên tiếp**, khác rỗng.
- **Vowel substring**: substring **chỉ gồm nguyên âm** (`a, e, i, o, u`) **và chứa đủ cả 5** nguyên âm.

Cho `word`, đếm số vowel substring (đếm theo **vị trí**: hai substring giống chữ nhưng ở vị trí khác nhau tính là 2).

| Ví dụ | word | Output | Giải thích |
|---|---|---|---|
| 1 | `"aeiouu"` | 2 | `"aeiou"` (0..4) và `"aeiouu"` (0..5) |
| 2 | `"unicornarihan"` | 0 | không đoạn nguyên âm nào đủ 5 loại |
| 3 | `"cuaieuouac"` | 7 | xem trace bên dưới |

Constraints: `1 <= len(word) <= 100`, chỉ chữ thường.

Chú ý: `"eiouu"` (1..5) **không** tính vì thiếu `a`.

In [1]:
word = "aeiouu"
VOWELS = set("aeiou")
for i in range(len(word)):
    for j in range(i, len(word)):
        sub = word[i:j + 1]
        if set(sub) == VOWELS:
            print(f"word[{i}:{j + 1}] = {sub!r}")

word[0:5] = 'aeiou'
word[0:6] = 'aeiouu'


## 2. Nền tảng Python

### 2.1. Slicing `word[i:j]` và số lượng substring

`word[i:j]` lấy ký tự từ chỉ số `i` đến **trước** `j`. Substring kết thúc **tại** `j` (bao gồm) là `word[i:j+1]`. Một chuỗi dài `n` có `n(n+1)/2` substring khác rỗng.

In [2]:
w = "abc"
subs = [w[i:j + 1] for i in range(len(w)) for j in range(i, len(w))]
print(subs, len(subs), "=", len(w) * (len(w) + 1) // 2)

['a', 'ab', 'abc', 'b', 'bc', 'c'] 6 = 6


List comprehension có **hai vòng `for`** đọc từ trái sang phải giống vòng lồng nhau: `for i ...` ở ngoài, `for j ...` ở trong.

### 2.2. `set` của một chuỗi, so sánh tập hợp

- `set("cuaie")` → các ký tự khác nhau `{'c','u','a','i','e'}`.
- `A <= B`: A là **tập con** của B (mọi phần tử của A đều thuộc B).
- `A == B`: hai tập bằng nhau.

"Chỉ gồm nguyên âm **và** đủ 5 nguyên âm" ⇔ `set(sub) == {'a','e','i','o','u'}`.

In [3]:
V = set("aeiou")
for sub in ["aeiou", "aeiouu", "eiouu", "caeiou", "uoiea"]:
    s = set(sub)
    print(f"{sub!r:<9} chỉ nguyên âm? {s <= V!s:<5} đủ 5? {len(s & V) == 5!s:<5} -> hợp lệ? {s == V}")

'aeiou'   chỉ nguyên âm? True  đủ 5? True  -> hợp lệ? True
'aeiouu'  chỉ nguyên âm? True  đủ 5? True  -> hợp lệ? True
'eiouu'   chỉ nguyên âm? True  đủ 5? False -> hợp lệ? False
'caeiou'  chỉ nguyên âm? False đủ 5? True  -> hợp lệ? False
'uoiea'   chỉ nguyên âm? True  đủ 5? True  -> hợp lệ? True


### 2.3. `in` với chuỗi nguyên âm

`ch in "aeiou"` kiểm tra ký tự có phải nguyên âm. Với chuỗi 5 ký tự thì nhanh như set.

In [4]:
print([ch for ch in "cuaieuouac" if ch in "aeiou"])
print(["V" if ch in "aeiou" else "-" for ch in "cuaieuouac"])

['u', 'a', 'i', 'e', 'u', 'o', 'u', 'a']
['-', 'V', 'V', 'V', 'V', 'V', 'V', 'V', 'V', '-']


### 2.4. `dict` lưu "vị trí xuất hiện cuối cùng"

`last[ch] = j` ghi đè mỗi khi gặp lại `ch` → luôn giữ vị trí **mới nhất**. `min(last.values())` cho vị trí xa nhất về bên trái trong số các "lần cuối".

In [5]:
last = {}
for j, ch in enumerate("aeiaou"):
    last[ch] = j
    print(f"j={j}, ch={ch}: last={last}")
print("min(last.values()) =", min(last.values()))

j=0, ch=a: last={'a': 0}
j=1, ch=e: last={'a': 0, 'e': 1}
j=2, ch=i: last={'a': 0, 'e': 1, 'i': 2}
j=3, ch=a: last={'a': 3, 'e': 1, 'i': 2}
j=4, ch=o: last={'a': 3, 'e': 1, 'i': 2, 'o': 4}
j=5, ch=u: last={'a': 3, 'e': 1, 'i': 2, 'o': 4, 'u': 5}
min(last.values()) = 1


## 3. Cách 1 — Brute force `O(n³)`: thử mọi substring

Hai vòng chọn `(i, j)`, với mỗi cặp tạo `word[i:j+1]` và `set(...)` — bước này tốn `O(độ dài)`. Tổng `O(n³)`. Với `n = 100`: khoảng 170.000 bước — vẫn qua, nhưng là **oracle** tốt để kiểm tra.

In [6]:
def count_vowel_brute(word):
    V = set("aeiou")
    total = 0
    n = len(word)
    for i in range(n):
        for j in range(i, n):
            if set(word[i:j + 1]) == V:
                total += 1
    return total

print(count_vowel_brute("aeiouu"))           # 2
print(count_vowel_brute("unicornarihan"))    # 0
print(count_vowel_brute("cuaieuouac"))       # 7

2
0
7


## 4. Nút thắt 1 → Cách 2 `O(n²)`: mở rộng dần và dừng khi gặp phụ âm

Hai quan sát (đúng như gợi ý của đề):

1. Cố định `i`, khi `j` tăng thì `word[i:j+1]` chỉ **thêm một ký tự**. Thay vì tạo lại set từ đầu, ta **thêm ký tự đó vào set** hiện có → mỗi bước `O(1)`.
2. Nếu `word[j]` là **phụ âm**, mọi substring bắt đầu tại `i` và kéo dài qua `j` đều **không hợp lệ** → `break` ngay.

In [7]:
def count_vowel_extend(word):
    total = 0
    n = len(word)
    for i in range(n):
        seen = set()
        for j in range(i, n):
            if word[j] not in "aeiou":
                break                      # gặp phụ âm: không mở rộng nữa
            seen.add(word[j])
            if len(seen) == 5:
                total += 1
    return total

print(count_vowel_extend("aeiouu"), count_vowel_extend("unicornarihan"), count_vowel_extend("cuaieuouac"))

2 0 7


### TRACE cách 2 trên `"cuaieuouac"`

Chỉ số: `c0 u1 a2 i3 e4 u5 o6 u7 a8 c9`. Đoạn nguyên âm là chỉ số 1..8.

| i | các j hợp lệ (seen đủ 5) | đếm |
|---|---|---|
| 0 | `c` là phụ âm → break ngay | 0 |
| 1 | seen đủ 5 khi j = 6 (`uaieuo`); j = 7, 8 vẫn đủ | 3 |
| 2 | đủ tại j = 6 (`aieuo`), j = 7, 8 | 3 |
| 3 | `ieuoua` đủ 5 khi j = 8 | 1 |
| 4..8 | thiếu `i` (chỉ số 3 nằm bên trái) | 0 |

Tổng: 3 + 3 + 1 = **7**.

In [8]:
def trace_extend(word):
    total = 0
    for i in range(len(word)):
        seen, hits = set(), []
        for j in range(i, len(word)):
            if word[j] not in "aeiou":
                break
            seen.add(word[j])
            if len(seen) == 5:
                hits.append(word[i:j + 1])
        total += len(hits)
        print(f"i={i} ({word[i]}): {len(hits)} substring {hits}")
    print("Tổng:", total)
    return total

trace_extend("cuaieuouac")

i=0 (c): 0 substring []
i=1 (u): 3 substring ['uaieuo', 'uaieuou', 'uaieuoua']
i=2 (a): 3 substring ['aieuo', 'aieuou', 'aieuoua']
i=3 (i): 1 substring ['ieuoua']
i=4 (e): 0 substring []
i=5 (u): 0 substring []
i=6 (o): 0 substring []
i=7 (u): 0 substring []
i=8 (a): 0 substring []
i=9 (c): 0 substring []
Tổng: 7


7

## 5. Nút thắt 2 → Insight cho `O(n)`

Cách 2 vẫn `O(n²)` khi chuỗi toàn nguyên âm (mỗi `i` có thể duyệt tới cuối). Đổi góc nhìn: **cố định điểm kết thúc `j`**, hỏi "có bao nhiêu điểm bắt đầu `i` hợp lệ?"

Với `j` là nguyên âm, gọi `start` = vị trí bắt đầu của **đoạn nguyên âm liên tiếp** chứa `j` (ngay sau phụ âm gần nhất). Điểm bắt đầu `i` phải:

- `i >= start` (không chứa phụ âm), và
- đoạn `[i, j]` chứa **đủ 5** nguyên âm ⇔ `i <= last[v]` với **mọi** nguyên âm `v` (lần xuất hiện cuối của `v` trước/tại `j` phải nằm trong đoạn) ⇔ `i <= min(last[a], last[e], last[i], last[o], last[u])`.

Vậy số `i` hợp lệ = `min_last - start + 1` (nếu cả 5 nguyên âm đã xuất hiện kể từ `start`), ngược lại 0.

In [9]:
# Minh họa tại j = 7 của "cuaieuouac"
word, j, start = "cuaieuouac", 7, 1
last = {}
for t in range(start, j + 1):
    last[word[t]] = t
print("last tại j=7:", last)
m = min(last.values())
print("min_last =", m, "-> các i hợp lệ:", list(range(start, m + 1)),
      "->", [word[i:j + 1] for i in range(start, m + 1)])

last tại j=7: {'u': 7, 'a': 2, 'i': 3, 'e': 4, 'o': 6}
min_last = 2 -> các i hợp lệ: [1, 2] -> ['uaieuou', 'aieuou']


## 6. Cách 3 (lời giải chính) — `O(n)` với `start` và `last`

Quy trình duyệt `j` từ trái sang phải:

1. Nếu `word[j]` là phụ âm: đặt `start = j + 1`, **xóa** `last` (đoạn nguyên âm mới bắt đầu sau phụ âm).
2. Ngược lại: `last[word[j]] = j`; nếu `len(last) == 5` thì cộng `min(last.values()) - start + 1`.

`min` trên dict 5 phần tử là `O(1)` (hằng số 5).

In [10]:
def count_vowel_linear(word):
    total = 0
    start = 0
    last = {}
    for j, ch in enumerate(word):
        if ch not in "aeiou":
            start = j + 1
            last.clear()                 # dict.clear(): xóa hết phần tử
            continue
        last[ch] = j
        if len(last) == 5:
            total += min(last.values()) - start + 1
    return total

print(count_vowel_linear("aeiouu"), count_vowel_linear("unicornarihan"), count_vowel_linear("cuaieuouac"))

2 0 7


## 7. TRACE cách 3 — bảng trạng thái

`word = "cuaieuouac"`:

| j | ch | start | last sau cập nhật | đủ 5? | min_last | cộng | total |
|---|---|---|---|---|---|---|---|
| 0 | c | 1 | {} | – | – | 0 | 0 |
| 1 | u | 1 | {u:1} | ✗ | – | 0 | 0 |
| 2 | a | 1 | {u:1,a:2} | ✗ | – | 0 | 0 |
| 3 | i | 1 | {u:1,a:2,i:3} | ✗ | – | 0 | 0 |
| 4 | e | 1 | {u:1,a:2,i:3,e:4} | ✗ | – | 0 | 0 |
| 5 | u | 1 | {u:5,a:2,i:3,e:4} | ✗ | – | 0 | 0 |
| 6 | o | 1 | {u:5,a:2,i:3,e:4,o:6} | ✓ | 2 | 2−1+1 = 2 | 2 |
| 7 | u | 1 | {u:7,a:2,i:3,e:4,o:6} | ✓ | 2 | 2 | 4 |
| 8 | a | 1 | {u:7,a:8,i:3,e:4,o:6} | ✓ | 3 | 3 | 7 |
| 9 | c | 10 | {} | – | – | 0 | 7 |

In [11]:
def trace_linear(word):
    total, start, last = 0, 0, {}
    for j, ch in enumerate(word):
        if ch not in "aeiou":
            start, last = j + 1, {}
            print(f"j={j} '{ch}': phụ âm -> start={start}, last={{}}")
            continue
        last[ch] = j
        if len(last) == 5:
            m = min(last.values())
            add = m - start + 1
            total += add
            print(f"j={j} '{ch}': last={last}, min_last={m}, cộng {add}, total={total}")
        else:
            print(f"j={j} '{ch}': last={last}, chưa đủ 5")
    return total

trace_linear("cuaieuouac")

j=0 'c': phụ âm -> start=1, last={}
j=1 'u': last={'u': 1}, chưa đủ 5
j=2 'a': last={'u': 1, 'a': 2}, chưa đủ 5
j=3 'i': last={'u': 1, 'a': 2, 'i': 3}, chưa đủ 5
j=4 'e': last={'u': 1, 'a': 2, 'i': 3, 'e': 4}, chưa đủ 5
j=5 'u': last={'u': 5, 'a': 2, 'i': 3, 'e': 4}, chưa đủ 5
j=6 'o': last={'u': 5, 'a': 2, 'i': 3, 'e': 4, 'o': 6}, min_last=2, cộng 2, total=2
j=7 'u': last={'u': 7, 'a': 2, 'i': 3, 'e': 4, 'o': 6}, min_last=2, cộng 2, total=4
j=8 'a': last={'u': 7, 'a': 8, 'i': 3, 'e': 4, 'o': 6}, min_last=3, cộng 3, total=7
j=9 'c': phụ âm -> start=10, last={}


7

## 8. Cách 4 — "Đúng 5 loại" = "Tối đa 5" − "Tối đa 4" (sliding window)

Kỹ thuật tổng quát cho bài "đếm substring có **đúng K** loại ký tự": `exactly(K) = at_most(K) − at_most(K−1)`. Hàm `at_most(K)` dùng **cửa sổ trượt** (sliding window) hai con trỏ `left`, `right`:

- Mở rộng `right`; nếu gặp phụ âm thì "đặt lại" cửa sổ sau nó.
- Nếu số loại nguyên âm trong cửa sổ > K thì tăng `left` (bỏ ký tự trái) cho đến khi ≤ K.
- Mọi substring kết thúc tại `right` và bắt đầu trong `[left, right]` đều có ≤ K loại → cộng `right − left + 1`.

In [12]:
from collections import defaultdict

def at_most_k_vowel_kinds(word, k):
    total = 0
    left = 0
    freq = defaultdict(int)          # dict có giá trị mặc định 0
    for right, ch in enumerate(word):
        if ch not in "aeiou":
            left = right + 1
            freq.clear()
            continue
        freq[ch] += 1
        while len(freq) > k:
            c = word[left]
            freq[c] -= 1
            if freq[c] == 0:
                del freq[c]          # xóa key khi hết, để len(freq) = số loại
            left += 1
        total += right - left + 1
    return total

def count_vowel_window(word):
    return at_most_k_vowel_kinds(word, 5) - at_most_k_vowel_kinds(word, 4)

print(count_vowel_window("aeiouu"), count_vowel_window("unicornarihan"), count_vowel_window("cuaieuouac"))

2 0 7


`defaultdict(int)`: khi truy cập key chưa có, tự tạo với giá trị `int()` = 0, nên `freq[ch] += 1` không lỗi. `del freq[c]` xóa key.

Vì sao trừ được? Mọi substring toàn nguyên âm có số loại từ 1..5. "≤ 5 loại" trừ "≤ 4 loại" còn lại đúng những substring có **5 loại**.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def countVowelSubstrings(self, word: str) -> int:
        total = 0
        start = 0
        last = {}
        for j, ch in enumerate(word):
            if ch not in "aeiou":
                start = j + 1
                last = {}
                continue
            last[ch] = j
            if len(last) == 5:
                total += min(last.values()) - start + 1
        return total

print(Solution().countVowelSubstrings("cuaieuouac"))

7


Với `n ≤ 100`, bản `O(n²)` (cách 2) cũng nộp rất ổn và dễ nghĩ ra trong phỏng vấn:

In [14]:
class SolutionQuadratic:
    def countVowelSubstrings(self, word: str) -> int:
        total = 0
        for i in range(len(word)):
            seen = set()
            for j in range(i, len(word)):
                if word[j] not in "aeiou":
                    break
                seen.add(word[j])
                total += len(seen) == 5
        return total

print(SolutionQuadratic().countVowelSubstrings("aeiouu"))

2


## 10. Vì sao cách 3 đúng?

Cố định `j` là nguyên âm. **Invariant:** trước khi xử lý `j`, `start` = vị trí đầu của đoạn nguyên âm liên tiếp chứa `j`, và `last[v]` = vị trí lớn nhất `< j` của nguyên âm `v` trong `[start, j−1]` (nếu có). Sau khi gán `last[word[j]] = j`, `last` đúng cho `[start, j]`.

Một điểm bắt đầu `i` cho substring hợp lệ `[i, j]` ⇔:

- `[i, j]` không chứa phụ âm ⇔ `i >= start` (vì `[start, j]` toàn nguyên âm và `start − 1` là phụ âm hoặc ngoài chuỗi).
- `[i, j]` chứa nguyên âm `v` ⇔ có vị trí của `v` trong `[i, j]` ⇔ vị trí **lớn nhất** của `v` trong `[start, j]` là `>= i` ⇔ `last[v] >= i`.

Đủ 5 nguyên âm ⇔ `i <= min(last.values())` và cả 5 key phải có mặt. Số `i` trong `[start, min_last]` là `min_last − start + 1`. Cộng theo mọi `j` → mỗi substring hợp lệ được đếm **đúng một lần** (theo điểm kết thúc của nó).

## 11. Độ phức tạp

Gọi `n = len(word)` (≤ 100).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n³)` | `O(n)` (substring tạm) | oracle |
| 2. Mở rộng + break | `O(n²)` | `O(1)` (set ≤ 5 phần tử) | dễ nghĩ, đủ cho n ≤ 100 |
| 3. start + last | `O(n)` | `O(1)` (dict ≤ 5 key) | tối ưu |
| 4. at_most(5) − at_most(4) | `O(n)` (mỗi con trỏ đi tối đa n bước) | `O(1)` | kỹ thuật tổng quát "đúng K" |

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<36} PASS ({actual})")

cases = [
    ("ví dụ 1", "aeiouu", 2),
    ("ví dụ 2", "unicornarihan", 0),
    ("ví dụ 3", "cuaieuouac", 7),
    ("1 ký tự", "a", 0),
    ("đúng 5 nguyên âm", "aeiou", 1),
    ("đảo thứ tự", "uoiea", 1),
    ("bị phụ âm cắt", "aeixou", 0),
    ("hai đoạn tách biệt", "aeioubaeiou", 2),
    ("lặp nhiều", "aaeiouu", 4),
    ("toàn phụ âm", "bcdfg", 0),
]
fs = [count_vowel_brute, count_vowel_extend, count_vowel_linear, count_vowel_window,
      Solution().countVowelSubstrings, SolutionQuadratic().countVowelSubstrings]
for name, w, exp in cases:
    for f in fs:
        assert f(w) == exp, (f, w, f(w))
    check(name, Solution().countVowelSubstrings(w), exp)

ví dụ 1                              PASS (2)
ví dụ 2                              PASS (0)
ví dụ 3                              PASS (7)
1 ký tự                              PASS (0)
đúng 5 nguyên âm                     PASS (1)
đảo thứ tự                           PASS (1)
bị phụ âm cắt                        PASS (0)
hai đoạn tách biệt                   PASS (2)
lặp nhiều                            PASS (4)
toàn phụ âm                          PASS (0)


In [16]:
import random
random.seed(2062)
for _ in range(1500):
    w = "".join(random.choice("aeioub") for _ in range(random.randint(1, 30)))
    answers = {f(w) for f in fs}
    assert len(answers) == 1, (w, [f(w) for f in fs])
print("1500 chuỗi ngẫu nhiên: 6 cài đặt khớp nhau — PASS")

1500 chuỗi ngẫu nhiên: 6 cài đặt khớp nhau — PASS


Ca `"aaeiouu"` = 4: điểm kết thúc ở `o`(4)? chưa có `u`. Kết thúc tại `u`(5): bắt đầu 0 hoặc 1 (cần chứa `a` → `i <= 1`) → 2; tại `u`(6): tương tự 2. Tổng 4.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra "đủ 5 nguyên âm", quên "chỉ gồm nguyên âm"

In [17]:
def wrong_only_contains(word):
    total = 0
    for i in range(len(word)):
        for j in range(i, len(word)):
            if set("aeiou") <= set(word[i:j + 1]):     # chỉ kiểm tra chứa đủ 5
                total += 1
    return total

print("Sai :", wrong_only_contains("cuaieuouac"))     # đếm cả substring có 'c'
print("Đúng:", count_vowel_brute("cuaieuouac"))

Sai : 14
Đúng: 7


### Lỗi 2: Đếm substring **khác nhau về nội dung** thay vì theo vị trí

Dùng `set` để gom các substring hợp lệ rồi đếm `len` sẽ gộp các substring giống chữ ở vị trí khác nhau.

In [18]:
def wrong_distinct_content(word):
    found = set()
    for i in range(len(word)):
        for j in range(i, len(word)):
            sub = word[i:j + 1]
            if set(sub) == set("aeiou"):
                found.add(sub)
    return len(found)

print("Sai :", wrong_distinct_content("aeiouxaeiou"))   # 1 (gộp hai "aeiou")
print("Đúng:", count_vowel_brute("aeiouxaeiou"))         # 2

Sai : 1
Đúng: 2


### Lỗi 3: Cách 3 mà quên xóa `last` khi gặp phụ âm

Vị trí cũ trước phụ âm vẫn nằm trong `last`, làm `len(last) == 5` sai và `min_last − start + 1` có thể âm.

In [19]:
def wrong_no_reset(word):
    total, start, last = 0, 0, {}
    for j, ch in enumerate(word):
        if ch not in "aeiou":
            start = j + 1               # quên last.clear()
            continue
        last[ch] = j
        if len(last) == 5:
            total += min(last.values()) - start + 1
    return total

print("Sai :", wrong_no_reset("aeioxu"))   # 'a' ở trước 'x' vẫn bị tính -> số âm/khác 0
print("Đúng:", count_vowel_linear("aeioxu"))

Sai : -4
Đúng: 0


### Lỗi 4: Ở cách 2, dùng `continue` thay vì `break` khi gặp phụ âm

`continue` bỏ qua phụ âm rồi **vẫn mở rộng tiếp**, coi như substring "nhảy qua" phụ âm — sai vì substring phải liên tiếp.

## 14. Mẹo nhận dạng pattern

- "Đếm substring thỏa điều kiện về **tập ký tự**" + `n` nhỏ → cố định `i`, mở rộng `j`, cập nhật set/đếm tăng dần, `break` khi chắc chắn không thể hợp lệ nữa.
- "Đếm substring có **đúng K loại**" → `at_most(K) − at_most(K−1)` với sliding window (LeetCode 992 Subarrays with K Different Integers).
- "Substring chứa **đủ mọi** loại trong một tập" → cố định điểm kết thúc, số điểm bắt đầu = `min(last occurrence) − start + 1` (LeetCode 1358 Number of Substrings Containing All Three Characters).
- Có ký tự "phá vỡ" (phụ âm) → chia chuỗi thành **các đoạn** độc lập.

## 15. Biến thể

1. **Ít nhất K loại nguyên âm** (substring toàn nguyên âm): `at_most(5) − at_most(K−1)`.
2. **Substring chứa đủ `a`, `b`, `c`** (LeetCode 1358, không có ký tự "phá vỡ").
3. **Độ dài substring vowel ngắn nhất** chứa đủ 5 nguyên âm.

In [20]:
def count_at_least_k_kinds(word, k):
    return at_most_k_vowel_kinds(word, 5) - at_most_k_vowel_kinds(word, k - 1)

def count_abc(s):
    last = {"a": -1, "b": -1, "c": -1}
    total = 0
    for j, ch in enumerate(s):
        last[ch] = j
        total += min(last.values()) + 1        # số điểm bắt đầu 0..min_last
    return total

print(count_at_least_k_kinds("aeiouu", 5), count_at_least_k_kinds("aei", 2))
print(count_abc("abcabc"))   # LeetCode 1358 ví dụ: 10

2 3
10


Để ý `count_abc`: khởi tạo `last` = −1 nên khi chưa đủ 3 ký tự, `min + 1 = 0` — không cần `if`. Đây là cùng ý tưởng với cách 3, với `start = 0`.

## 16. Bài tập tự luyện

**Bài 1 — Tính tay:** liệt kê mọi vowel substring của `"aaeiou"` (ghi vị trí `[i, j]`).

**Bài 2 — Đoán output:** `set("aeiouu") == set("aeiou")`, `set("ab") <= set("abc")`, `"hello"[1:3]`, `len("abcd") * 5 // 2`.

**Bài 3 — TODO:** viết `exercise_count_extend(word)` (cách 2: mở rộng + break).

**Bài 4 — TODO:** `vowel_segments(word)` trả list các **đoạn nguyên âm liên tiếp tối đại** (vd `"cuaieuouac"` → `["uaieuoua"]`).

**Bài 5 — TODO biến thể:** `shortest_vowel_substring(word)` — độ dài ngắn nhất của vowel substring, hoặc `0` nếu không có.

**Bài 6 — Mini challenge:** viết lại cách 3 `exercise_linear(word)` mà **không dùng dict**: dùng 5 biến hoặc list `last = [-1]*5` và `"aeiou".index(ch)`.

In [21]:
def exercise_count_extend(word):
    # TODO: for i ...; seen = set(); for j ...: break nếu phụ âm; add; nếu đủ 5 thì cộng
    pass

# assert exercise_count_extend("cuaieuouac") == 7

In [22]:
def vowel_segments(word):
    # TODO
    pass

# assert vowel_segments("cuaieuouac") == ["uaieuoua"]
# assert vowel_segments("unicornarihan") == ["u", "i", "o", "a", "i", "a"]

In [23]:
def shortest_vowel_substring(word):
    # TODO
    pass

# assert shortest_vowel_substring("cuaieuouac") == 5
# assert shortest_vowel_substring("abc") == 0

In [24]:
def exercise_linear(word):
    # TODO: last = [-1] * 5 ; idx = "aeiou".index(ch)
    pass

# assert exercise_linear("cuaieuouac") == 7

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | cần chứa `u` ở cuối → j = 5 | điểm bắt đầu phải chứa `a`: i ∈ {0, 1} | `[0,5]`, `[1,5]` |
| 2 | set bỏ trùng | `<=` là tập con | `True`, `True`, `'el'`, `10` |
| 3 | xem mục 4 | `if word[j] not in "aeiou": break` | `if len(seen) == 5: total += 1` |
| 4 | gom ký tự nguyên âm liên tiếp | gặp phụ âm thì "chốt" đoạn hiện tại | nhớ chốt đoạn cuối sau vòng lặp |
| 5 | cố định `j`, `i` tốt nhất là `min_last` | độ dài `j − min_last + 1` | cập nhật `best = min(best, ...)` |
| 6 | ánh xạ nguyên âm → 0..4 | khi gặp phụ âm đặt lại `last = [-1]*5` | đủ 5 ⇔ `min(last) >= start` |

## 18. Đáp án tham khảo

**Bài 1:** `[0,5]` = `"aaeiou"` và `[1,5]` = `"aeiou"` → 2 substring.

**Bài 2:** `True` · `True` · `'el'` · `10`.

In [25]:
w = "aaeiou"
print([(i, j) for i in range(len(w)) for j in range(i, len(w)) if set(w[i:j + 1]) == set("aeiou")])
print(set("aeiouu") == set("aeiou"), set("ab") <= set("abc"), "hello"[1:3], len("abcd") * 5 // 2)

[(0, 5), (1, 5)]
True True el 10


In [26]:
def exercise_count_extend_answer(word):
    total = 0
    for i in range(len(word)):
        seen = set()
        for j in range(i, len(word)):
            if word[j] not in "aeiou":
                break
            seen.add(word[j])
            if len(seen) == 5:
                total += 1
    return total

def vowel_segments_answer(word):
    segs, cur = [], ""
    for ch in word:
        if ch in "aeiou":
            cur += ch
        else:
            if cur:
                segs.append(cur)
            cur = ""
    if cur:
        segs.append(cur)               # chốt đoạn cuối
    return segs

def shortest_vowel_substring_answer(word):
    best, start, last = 0, 0, {}
    for j, ch in enumerate(word):
        if ch not in "aeiou":
            start, last = j + 1, {}
            continue
        last[ch] = j
        if len(last) == 5:
            length = j - min(last.values()) + 1
            best = length if best == 0 else min(best, length)
    return best

def exercise_linear_answer(word):
    total, start, last = 0, 0, [-1] * 5
    for j, ch in enumerate(word):
        if ch not in "aeiou":
            start, last = j + 1, [-1] * 5
            continue
        last["aeiou".index(ch)] = j
        m = min(last)
        if m >= start:                  # cả 5 nguyên âm đã xuất hiện trong đoạn hiện tại
            total += m - start + 1
    return total

assert exercise_count_extend_answer("cuaieuouac") == 7
assert vowel_segments_answer("cuaieuouac") == ["uaieuoua"]
assert vowel_segments_answer("unicornarihan") == ["u", "i", "o", "a", "i", "a"]
assert shortest_vowel_substring_answer("cuaieuouac") == 5
assert shortest_vowel_substring_answer("abc") == 0
assert exercise_linear_answer("cuaieuouac") == 7
random.seed(4)
for _ in range(1000):
    w = "".join(random.choice("aeiouxb") for _ in range(random.randint(1, 25)))
    assert exercise_linear_answer(w) == exercise_count_extend_answer(w) == count_vowel_brute(w)
    brute_short = min((j - i + 1 for i in range(len(w)) for j in range(i, len(w))
                       if set(w[i:j + 1]) == set("aeiou")), default=0)
    assert shortest_vowel_substring_answer(w) == brute_short
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


`min(iterable, default=0)`: nếu iterable rỗng thì trả `0` thay vì báo lỗi.

## 19. Tổng kết — template + checklist

**Template "đếm substring chứa đủ mọi loại" (cố định điểm kết thúc):**

```python
total, start, last = 0, 0, {}
for j, ch in enumerate(s):
    if ch là ký tự phá vỡ:
        start, last = j + 1, {}
        continue
    last[ch] = j
    if len(last) == SỐ_LOẠI:
        total += min(last.values()) - start + 1
```

**Checklist submit**

- [ ] Substring phải **toàn nguyên âm** (gặp phụ âm thì cắt / `break`).
- [ ] Phải có **đủ 5** nguyên âm.
- [ ] Đếm theo **vị trí**, không gộp substring trùng nội dung.
- [ ] Ở cách `O(n)`: nhớ đặt lại `start` **và** xóa `last` khi gặp phụ âm.
- [ ] Tên method `countVowelSubstrings`, trả `int`.